In [1]:
import polars as pl

In [2]:
import polars as pl

inicial = (
    pl.scan_parquet(r"C:\Users\diogo.durao\Documents\covid_sp.parquet")
    .select([
        "paciente_idade",
        "paciente_dataNascimento",
        "paciente_enumSexoBiologico",
        "paciente_endereco_nmMunicipio",
        "paciente_endereco_uf",
        "vacina_dataAplicacao",
        "vacina_fabricante_nome",
        "vacina_nome"
    ])
    .with_columns(
        pl.col("paciente_idade").cast(pl.Int64, strict=False),
        pl.col("paciente_dataNascimento").str.to_date("%Y-%m-%d", strict=False),
        pl.col("vacina_dataAplicacao").str.to_date("%Y-%m-%d", strict=False)
    )
)

display(inicial.head(5).collect())

paciente_idade,paciente_dataNascimento,paciente_enumSexoBiologico,paciente_endereco_nmMunicipio,paciente_endereco_uf,vacina_dataAplicacao,vacina_fabricante_nome,vacina_nome
i64,date,str,str,str,date,str,str
32,1989-05-29,"""F""","""GUARULHOS""","""SP""",2022-04-05,"""JANSSEN""","""COVID-19 JANSSEN - Ad26.COV2.S"""
59,1961-09-06,"""F""","""IBITINGA""","""SP""",2021-05-31,"""ASTRAZENECA/FIOCRUZ""","""COVID-19 ASTRAZENECA/FIOCRUZ -…"
85,1936-03-09,"""M""","""SAO PAULO""","""SP""",2021-12-04,"""PFIZER""","""COVID-19 PFIZER - COMIRNATY"""
35,1985-11-22,"""M""","""SAO PAULO""","""SP""",2021-08-14,"""SINOVAC/BUTANTAN""","""COVID-19 SINOVAC/BUTANTAN - CO…"
53,1968-05-18,"""M""","""SAO PAULO""","""SP""",2021-09-14,"""PFIZER""","""COVID-19 PFIZER - COMIRNATY"""


In [4]:
# 1. Calcular os limites (Q1 e Q3) baseados na quantidade de vacinas por município

df_municipios = inicial.group_by(
    "paciente_endereco_nmMunicipio",
    "paciente_endereco_uf"
).agg(
    pl.len().alias("Qtd_Vacinas")
)

# Precisamos coletar para calcular os quartis exatos da distribuição das cidades
df_agregado = df_municipios.collect()

q1 = df_agregado.select(pl.col("Qtd_Vacinas").quantile(0.25)).item()
q3 = df_agregado.select(pl.col("Qtd_Vacinas").quantile(0.75)).item()

print(f"Municípios abaixo de {q1:.2f} vacinas são 'Baixa Vacinação'")
print(f"Municípios acima de {q3:.2f} vacinas são 'Alta Vacinação'")

# 2. Criar a coluna "Etiqueta" (Classificação) e filtrar apenas os extremos

df_export = df_agregado.with_columns(
    pl.when(pl.col("Qtd_Vacinas") > q3).then(pl.lit("Alta Vacinação"))
    .when(pl.col("Qtd_Vacinas") < q1).then(pl.lit("Baixa Vacinação"))
    .otherwise(pl.lit("Médio"))
    .alias("Etiqueta_Classificacao")
).filter(
    (pl.col("Etiqueta_Classificacao") == "Alta Vacinação") |
    (pl.col("Etiqueta_Classificacao") == "Baixa Vacinação")
)

# 3. Exportar para CSV
df_export.write_csv("vacinas_extremos_municipios.csv")

print("Arquivo 'vacinas_extremos_municipios.csv' gerado com sucesso!")
print(df_export.head())

Municípios abaixo de 27.00 vacinas são 'Baixa Vacinação'
Municípios acima de 499.00 vacinas são 'Alta Vacinação'
Arquivo 'vacinas_extremos_municipios.csv' gerado com sucesso!
shape: (5, 4)
┌───────────────────────────────┬──────────────────────┬─────────────┬────────────────────────┐
│ paciente_endereco_nmMunicipio ┆ paciente_endereco_uf ┆ Qtd_Vacinas ┆ Etiqueta_Classificacao │
│ ---                           ┆ ---                  ┆ ---         ┆ ---                    │
│ str                           ┆ str                  ┆ u32         ┆ str                    │
╞═══════════════════════════════╪══════════════════════╪═════════════╪════════════════════════╡
│ FERNANDOPOLIS                 ┆ SP                   ┆ 49376       ┆ Alta Vacinação         │
│ TAPIRAI                       ┆ MG                   ┆ 3           ┆ Baixa Vacinação        │
│ PIEDADE                       ┆ SP                   ┆ 36052       ┆ Alta Vacinação         │
│ BARRINHA                      ┆ SP       